In [3]:
%%configure -f
{"conf": {"spark.pyspark.python": "python3",
          "spark.pyspark.virtualenv.enabled": "true",
          "spark.pyspark.virtualenv.type": "native",
          "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"}}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1794820524138_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1794820524138_0002,pyspark,idle,Link,Link,None,✔


# INF2006 G12 - Task 4: Data analytics (volume anomalies, news signal)

**Inputs (Task 2 outputs, HDFS):** `/corpus/cleaned/prices/` (605,887 daily bars) and
`/corpus/cleaned/gdelt/daily_features/` (150 daily GDELT aggregates keyed by `available_date`, i.e. DATEADDED).

**Questions**
- **Q1.** When is trading activity for a security unusually high or low versus its *own* history, and how does that relate to price movement?
- **Q2.** Do GDELT news-volume / tone groups differ in subsequent market volatility, and does news add anything beyond today's volatility?

**Tools.** All heavy work (window functions, joins, group aggregates) runs in PySpark on the full tables. Only small aggregated
result tables (<= ~1,300 rows) are converted to pandas for charts and a Spearman check.

**Run order.** Run every cell top to bottom. Outputs go to `OUT_DIR` (default `evidence/task4/`) with fixed file names.

**Kernel note.** The notebook expects `spark` to exist in the same Python process that draws charts. If your kernel runs Spark remotely
(sparkmagic/Livy), plots may need `%matplot plt` or a local kernel; see the README note in the report appendix.

In [4]:
sc.install_pypi_package("numpy")
sc.install_pypi_package("pandas")
sc.install_pypi_package("matplotlib")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1794820524138_0002/container_1794820524138_0002_01_000001/tmp/spark-9f3034f3-c92a-4344-a776-3fe1a7b9e6e3
    Can't uninstall 'python-dateutil'. No files were found to uninstall.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.2.3 requires tzdata>=2022.7, which is not installed.
matplotlib 3.9.4 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.9.4 requires cycler>=0.10, which is not installed.
matplotlib 3.9.4 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.9.4 requires importlib-resources>=3.2.0; python_version < "3.10", which is not installed.
matplotlib 3.9.4 requires kiwisolver>=1.3.1

In [5]:
import os, json, time, datetime
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
T0 = time.time()
TIMINGS = {}

try:
    spark                      # provided by the EMR kernel
except NameError:              # local test only
    spark = (SparkSession.builder.master("local[2]").appName("g12_task4")
             .config("spark.ui.enabled", "false").getOrCreate())

ON_CLUSTER = not spark.sparkContext.master.startswith("local")
spark.conf.set("spark.sql.shuffle.partitions", "32" if ON_CLUSTER else "8")

# ---- parameters (change here only) ----
PRICES_PATH = "s3://inf2006-g12-2500837/task2/cleaned/prices"
GDELT_PATH  = "s3://inf2006-g12-2500837/task2/cleaned/gdelt/daily_features"
OUT_DIR     = os.environ.get("G12_OUT_DIR", "evidence/task4")
WINDOW      = 20      # trading days of own history for the volume baseline
Z_HIGH, Z_LOW = 2.0, -2.0
MAX_ABS_RET = 0.5     # |daily return| above this is treated as a probable split/data artefact (set to NULL)
N_GROUPS    = 3       # terciles for news groups
os.makedirs(OUT_DIR, exist_ok=True)

def stage(name, t_start):
    TIMINGS[name] = round(time.time() - t_start, 1)
    print(f"[{name}] {TIMINGS[name]} s")

print("RUN_ID", RUN_ID, "| Spark", spark.version, "| on cluster:", ON_CLUSTER)
print("PRICES_PATH", PRICES_PATH); print("GDELT_PATH ", GDELT_PATH); print("OUT_DIR    ", OUT_DIR)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RUN_ID 20261116T092814Z | Spark 3.5.5-amzn-1 | on cluster: True
PRICES_PATH s3://inf2006-g12-2500837/task2/cleaned/prices
GDELT_PATH  s3://inf2006-g12-2500837/task2/cleaned/gdelt/daily_features
OUT_DIR     evidence/task4

## 0. Load inputs and check scope

In [6]:
t = time.time()
prices = spark.read.parquet(PRICES_PATH).select(
    "ticker", "trade_date", "open_price", "high_price", "low_price", "close_price", "volume")
gdelt = spark.read.parquet(GDELT_PATH)

n_prices, n_gdelt = prices.count(), gdelt.count()
scope = prices.agg(F.min("trade_date").alias("d0"), F.max("trade_date").alias("d1"),
                   F.countDistinct("ticker").alias("tickers"),
                   F.countDistinct("trade_date").alias("trading_days")).first()
g_scope = gdelt.agg(F.min("available_date").alias("d0"), F.max("available_date").alias("d1")).first()

EXPECT = {"price_rows": 605887, "gdelt_rows": 150}
print("price rows  :", n_prices, "PASS" if n_prices == EXPECT["price_rows"] else "FAIL (expected 605,887)")
print("gdelt rows  :", n_gdelt,  "PASS" if n_gdelt  == EXPECT["gdelt_rows"]  else "FAIL (expected 150)")
print("prices span :", scope["d0"], "to", scope["d1"], "|", scope["tickers"], "tickers |", scope["trading_days"], "trading days")
print("gdelt span  :", g_scope["d0"], "to", g_scope["d1"], "(three blocks: Jul-Sep 2015, Jan 2016, Jan 2017)")
stage("load_and_check", t)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

price rows  : 605887 PASS
gdelt rows  : 150 PASS
prices span : 2013-02-08 to 2017-12-29 | 505 tickers | 1233 trading days
gdelt span  : 2015-07-01 to 2017-01-30 (three blocks: Jul-Sep 2015, Jan 2016, Jan 2017)
[load_and_check] 10.8 s

## Q1. Unusual trading activity versus a security's own history

Method: for each ticker, `z = (log1p(volume) - mean) / std`, where mean and std use only the **previous 20 trading rows**
(today excluded, so a spike cannot inflate its own baseline). Log volume is used because raw volume is strongly right-skewed.
Days are labelled **high** (z >= 2), **low** (z <= -2) or **normal**. Daily returns with |return| > 50% are set to NULL
(probable splits, counted below) but the day stays in the volume baseline.

Tool choice: PySpark window functions, because the analysis needs per-ticker rolling statistics over 605,887 rows.

In [7]:
t = time.time()
w_t    = Window.partitionBy("ticker").orderBy("trade_date")
w_hist = w_t.rowsBetween(-WINDOW, -1)

base = (prices
    .withColumn("log_vol", F.log1p(F.col("volume").cast("double")))
    .withColumn("ret", F.col("close_price") / F.lag("close_price").over(w_t) - 1)
    .withColumn("abs_ret", F.when(F.abs("ret") <= MAX_ABS_RET, F.abs("ret")))
    .withColumn("next_abs_ret", F.lead("abs_ret").over(w_t))
    .withColumn("range_pct", (F.col("high_price") - F.col("low_price")) / F.col("close_price"))
    .withColumn("n_hist", F.count("log_vol").over(w_hist))
    .withColumn("mu", F.avg("log_vol").over(w_hist))
    .withColumn("sd", F.stddev("log_vol").over(w_hist))
    .withColumn("z", F.when((F.col("n_hist") == WINDOW) & (F.col("sd") > 0),
                            (F.col("log_vol") - F.col("mu")) / F.col("sd")))
    .withColumn("vol_class", F.when(F.col("z") >= Z_HIGH, "high")
                              .when(F.col("z") <= Z_LOW, "low")
                              .when(F.col("z").isNotNull(), "normal"))
    .cache())

n_base = base.count()
n_artefact = base.filter(F.abs("ret") > MAX_ABS_RET).count()
n_scored = base.filter(F.col("z").isNotNull()).count()
print("rows:", n_base, "| scored (>= %d prior days):" % WINDOW, n_scored,
      "| |return|>%.0f%% set to NULL:" % (MAX_ABS_RET * 100), n_artefact)

q1_summary = (base.filter(F.col("vol_class").isNotNull())
    .groupBy("vol_class")
    .agg(F.count("*").alias("n_days"),
         F.avg("abs_ret").alias("mean_abs_ret_same_day"),
         F.expr("percentile_approx(abs_ret, 0.5)").alias("median_abs_ret_same_day"),
         F.avg("next_abs_ret").alias("mean_abs_ret_next_day"),
         F.avg("range_pct").alias("mean_intraday_range"))
    .withColumn("share_of_scored_days", F.col("n_days") / F.lit(n_scored))
    .toPandas().set_index("vol_class").loc[["high", "normal", "low"]].reset_index())

q1_bucket = (base.filter(F.col("z").isNotNull())
    .withColumn("z_bucket", F.floor(F.greatest(F.least(F.col("z"), F.lit(4.0)), F.lit(-3.0)) * 2) / 2)
    .groupBy("z_bucket")
    .agg(F.count("*").alias("n_days"),
         F.avg("abs_ret").alias("mean_abs_ret_same_day"),
         F.avg("next_abs_ret").alias("mean_abs_ret_next_day"))
    .orderBy("z_bucket").toPandas())

q1_monthly = (base.filter(F.col("z").isNotNull())
    .withColumn("month", F.date_trunc("month", F.col("trade_date")).cast("date"))
    .groupBy("month")
    .agg(F.count("*").alias("n_scored_days"),
         F.sum((F.col("vol_class") == "high").cast("int")).alias("n_high"),
         F.sum((F.col("vol_class") == "low").cast("int")).alias("n_low"))
    .withColumn("share_high", F.col("n_high") / F.col("n_scored_days"))
    .orderBy("month").toPandas())
stage("q1_compute", t)
q1_summary

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

rows: 605887 | scored (>= 20 prior days): 595789 | |return|>50% set to NULL: 6
[q1_compute] 13.1 s
  vol_class  n_days  ...  mean_intraday_range  share_of_scored_days
0      high   32508  ...             0.033276              0.054563
1    normal  545647  ...             0.018370              0.915839
2       low   17634  ...             0.012803              0.029598

[3 rows x 7 columns]

In [8]:
normal_ret = float(q1_summary.loc[q1_summary.vol_class == "normal", "mean_abs_ret_same_day"].iloc[0])
q1_summary["same_day_abs_ret_vs_normal"] = q1_summary["mean_abs_ret_same_day"] / normal_ret

q1_summary.to_csv(f"{OUT_DIR}/q1_class_summary.csv", index=False)
q1_bucket.to_csv(f"{OUT_DIR}/q1_zbucket_summary.csv", index=False)
q1_monthly.to_csv(f"{OUT_DIR}/q1_monthly_flags.csv", index=False)
print(q1_summary.round(4).to_string(index=False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

vol_class  n_days  mean_abs_ret_same_day  median_abs_ret_same_day  mean_abs_ret_next_day  mean_intraday_range  share_of_scored_days  same_day_abs_ret_vs_normal
     high   32508                 0.0255                   0.0179                 0.0148               0.0333                0.0546                      2.5760
   normal  545647                 0.0099                   0.0073                 0.0104               0.0184                0.9158                      1.0000
      low   17634                 0.0053                   0.0040                 0.0085               0.0128                0.0296                      0.5387

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
order = q1_summary["vol_class"].tolist()
bars = ax[0].bar(order, q1_summary["mean_abs_ret_same_day"] * 100, color=["#c0392b", "#7f8c8d", "#2980b9"])
for b, n in zip(bars, q1_summary["n_days"]):
    ax[0].text(b.get_x() + b.get_width() / 2, b.get_height(), f"n={n:,}", ha="center", va="bottom", fontsize=9)
ax[0].set_ylabel("Mean |daily return| (%)"); ax[0].set_xlabel("Volume class (vs own prior 20 days)")
ax[0].set_title("Q1: same-day price move by volume class")

ax[1].plot(q1_bucket["z_bucket"], q1_bucket["mean_abs_ret_same_day"] * 100, marker="o", label="same day")
ax[1].plot(q1_bucket["z_bucket"], q1_bucket["mean_abs_ret_next_day"] * 100, marker="s", label="next day")
ax[1].set_xlabel("Volume z-score bucket (clipped to [-3, 4])"); ax[1].set_ylabel("Mean |daily return| (%)")
ax[1].set_title("Q1: price move rises with unusual volume"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q1_volume_class_and_zbucket.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(pd.to_datetime(q1_monthly["month"]), q1_monthly["share_high"] * 100)
ax.set_ylabel("% of scored days flagged high"); ax.set_xlabel("Month")
ax.set_title("Q1: share of ticker-days with unusually high volume, by month")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q1_monthly_high_share.png", dpi=150); plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Q2. Does news volume / tone help explain subsequent market volatility?

Method: (1) market volatility per trading day = average |daily return| across tickers (PySpark); (2) join to the 150 GDELT dates on
`trade_date = available_date`; (3) split the joined days into terciles of `event_count` (news volume) and of `avg_tone` (sentiment);
(4) compare **next-trading-day** volatility across groups, with group sizes and spread. News on day t is compared with volatility on t+1,
because some day-t articles are published after the close. Same-day volatility is also reported and labelled contemporaneous.
(5) To ask whether news adds anything beyond price data alone, compare groups within low/high *today* volatility and test the
Spearman correlation of news with next-day volatility after removing the part explained by today's volatility.

Tool choice: PySpark for the daily aggregation, join and group statistics; the ~100-row result is analysed in pandas.
Observed relationships only; no causal claim is made.

In [11]:
t = time.time()
w_d = Window.orderBy("trade_date")
mkt = (base.filter(F.col("abs_ret").isNotNull())
       .groupBy("trade_date")
       .agg(F.avg("abs_ret").alias("mkt_vol"), F.countDistinct("ticker").alias("n_tickers"))
       .withColumn("mkt_vol_next", F.lead("mkt_vol").over(w_d)))

news = gdelt.withColumnRenamed("available_date", "trade_date")
joined = (mkt.join(news, "trade_date", "inner")
          .filter(F.col("mkt_vol_next").isNotNull())
          .withColumn("news_group", F.ntile(N_GROUPS).over(Window.orderBy("event_count")))
          .withColumn("tone_group", F.ntile(N_GROUPS).over(Window.orderBy("avg_tone")))
          .withColumn("today_vol_group", F.ntile(2).over(Window.orderBy("mkt_vol")))
          .withColumn("block", F.year("trade_date"))      # each GDELT block falls in a different year
          .cache())
n_joined = joined.count()
print("market-days:", mkt.count(), "| GDELT dates:", n_gdelt,
      "| joined trading days with a next day:", n_joined,
      "| GDELT dates dropped (weekends/holidays):", n_gdelt - n_joined)

def group_stats(col):
    return (joined.groupBy(col)
        .agg(F.count("*").alias("n_days"),
             F.min("event_count" if col == "news_group" else "avg_tone").alias("group_min"),
             F.max("event_count" if col == "news_group" else "avg_tone").alias("group_max"),
             F.avg("mkt_vol_next").alias("mean_next_day_vol"),
             F.expr("percentile_approx(mkt_vol_next, 0.5)").alias("median_next_day_vol"),
             F.stddev("mkt_vol_next").alias("sd_next_day_vol"),
             F.avg("mkt_vol").alias("mean_same_day_vol_contemporaneous"))
        .orderBy(col).toPandas())

q2_news = group_stats("news_group")
q2_tone = group_stats("tone_group")
q2_controlled = (joined.groupBy("today_vol_group", "news_group")
    .agg(F.count("*").alias("n_days"), F.avg("mkt_vol_next").alias("mean_next_day_vol"))
    .orderBy("today_vol_group", "news_group").toPandas())
q2_block_mix = (joined.groupBy("block")
    .agg(F.count("*").alias("n_days"),
         F.avg("mkt_vol").alias("mean_same_day_vol"),
         F.avg("avg_tone").alias("mean_tone"),
         F.avg("event_count").alias("mean_event_count"),
         F.sum((F.col("tone_group") == 1).cast("int")).alias("n_in_most_negative_tone_group"),
         F.sum((F.col("tone_group") == 3).cast("int")).alias("n_in_least_negative_tone_group"))
    .orderBy("block").toPandas())
daily = joined.select("trade_date", "block", "mkt_vol", "mkt_vol_next", "event_count", "avg_tone",
                      "news_group", "tone_group", "today_vol_group").orderBy("trade_date").toPandas()
stage("q2_compute", t)
q2_news

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

market-days: 1232 | GDELT dates: 150 | joined trading days with a next day: 100 | GDELT dates dropped (weekends/holidays): 50
[q2_compute] 2.0 s
   news_group  n_days  ...  sd_next_day_vol  mean_same_day_vol_contemporaneous
0           1      34  ...         0.007553                           0.013036
1           2      33  ...         0.006563                           0.015455
2           3      33  ...         0.005615                           0.012582

[3 rows x 8 columns]

In [12]:
import numpy as np
try:
    from scipy.stats import spearmanr
except ImportError:
    spearmanr = None

def spearman(x, y):
    if spearmanr is not None:
        r, p = spearmanr(x, y); return float(r), float(p)
    return float(pd.Series(x).rank().corr(pd.Series(y).rank())), float("nan")

# part of next-day volatility NOT explained by today's volatility (simple linear fit)
X = np.column_stack([np.ones(len(daily)), daily["mkt_vol"].to_numpy()])
beta, *_ = np.linalg.lstsq(X, daily["mkt_vol_next"].to_numpy(), rcond=None)
daily["resid_next_vol"] = daily["mkt_vol_next"] - X @ beta
r2_today_only = 1 - daily["resid_next_vol"].var() / daily["mkt_vol_next"].var()

rows = []
for label, x, y in [
    ("event_count vs next-day vol",              daily["event_count"], daily["mkt_vol_next"]),
    ("avg_tone vs next-day vol",                 daily["avg_tone"],    daily["mkt_vol_next"]),
    ("event_count vs next-day vol | today vol",  daily["event_count"], daily["resid_next_vol"]),
    ("avg_tone vs next-day vol | today vol",     daily["avg_tone"],    daily["resid_next_vol"]),
    ("today vol vs next-day vol (volatility clustering)", daily["mkt_vol"], daily["mkt_vol_next"]),
    # within-block: remove each GDELT block's own average, so differences BETWEEN the three periods cannot drive the result
    ("WITHIN-BLOCK event_count vs next-day vol | today vol",
        daily["event_count"] - daily.groupby("block")["event_count"].transform("mean"),
        daily["resid_next_vol"] - daily.groupby("block")["resid_next_vol"].transform("mean")),
    ("WITHIN-BLOCK avg_tone vs next-day vol | today vol",
        daily["avg_tone"] - daily.groupby("block")["avg_tone"].transform("mean"),
        daily["resid_next_vol"] - daily.groupby("block")["resid_next_vol"].transform("mean")),
]:
    r, p = spearman(x, y)
    rows.append({"pair": label, "spearman_rho": round(r, 3), "p_value": round(p, 3), "n_days": len(daily)})
q2_corr = pd.DataFrame(rows)
print("R^2 of next-day vol explained by today's vol alone:", round(float(r2_today_only), 3))

q2_news.to_csv(f"{OUT_DIR}/q2_news_volume_groups.csv", index=False)
q2_tone.to_csv(f"{OUT_DIR}/q2_tone_groups.csv", index=False)
q2_controlled.to_csv(f"{OUT_DIR}/q2_controlled_by_today_vol.csv", index=False)
q2_corr.to_csv(f"{OUT_DIR}/q2_spearman_checks.csv", index=False)
daily.to_csv(f"{OUT_DIR}/q2_joined_daily.csv", index=False)
q2_block_mix.to_csv(f"{OUT_DIR}/q2_block_composition.csv", index=False)
print(q2_block_mix.round(4).to_string(index=False)); print()
print(q2_news.round(5).to_string(index=False)); print()
print(q2_tone.round(5).to_string(index=False)); print()
print(q2_controlled.round(5).to_string(index=False)); print()
print(q2_corr.to_string(index=False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

R^2 of next-day vol explained by today's vol alone: 0.192
 block  n_days  mean_same_day_vol  mean_tone  mean_event_count  n_in_most_negative_tone_group  n_in_least_negative_tone_group
  2015      62             0.0133    -2.0781       218681.8548                             16                              19
  2016      19             0.0188    -2.3617       222599.3158                             16                               0
  2017      19             0.0097    -1.8625       226461.4737                              2                              14

 news_group  n_days  group_min  group_max  mean_next_day_vol  median_next_day_vol  sd_next_day_vol  mean_same_day_vol_contemporaneous
          1      34     175886     213692            0.01333              0.01065          0.00755                            0.01304
          2      33     213969     227578            0.01488              0.01326          0.00656                            0.01546
          3      33     228323     

In [13]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for a, col, title, labels in [(ax[0], "news_group", "News volume (GDELT event_count)", ["low", "mid", "high"]),
                              (ax[1], "tone_group", "News tone (GDELT avg_tone)", ["most negative", "mid", "least negative"])]:
    data = [daily.loc[daily[col] == g, "mkt_vol_next"] * 100 for g in range(1, N_GROUPS + 1)]
    a.boxplot(data)
    a.set_xticklabels([f"{l}\n(n={len(d)})" for l, d in zip(labels, data)])
    a.set_xlabel(title + " tercile"); a.set_title("Next-day market volatility by group")
ax[0].set_ylabel("Next-day mean |return| across tickers (%)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q2_next_day_vol_by_news_group.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 4.2))
sc = ax.scatter(daily["mkt_vol"] * 100, daily["mkt_vol_next"] * 100, c=daily["event_count"], cmap="viridis", s=22)
plt.colorbar(sc, label="GDELT event_count")
ax.set_xlabel("Today's market volatility (%)"); ax.set_ylabel("Next-day market volatility (%)")
ax.set_title("Q2: volatility clustering dominates; colour = news volume")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q2_today_vs_next_day_vol.png", dpi=150); plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Run manifest (evidence for Appendix B)

In [14]:
manifest = {
    "run_id": RUN_ID, "spark_version": spark.version, "on_cluster": ON_CLUSTER,
    "inputs": {"prices": PRICES_PATH, "gdelt_daily_features": GDELT_PATH},
    "row_counts": {"prices": n_prices, "gdelt_dates": n_gdelt, "scored_ticker_days": n_scored,
                   "returns_nulled_as_artefacts": n_artefact, "q2_joined_days": n_joined},
    "parameters": {"window": WINDOW, "z_high": Z_HIGH, "z_low": Z_LOW, "max_abs_ret": MAX_ABS_RET, "groups": N_GROUPS},
    "stage_seconds": TIMINGS, "total_seconds": round(time.time() - T0, 1),
    "outputs": sorted(os.listdir(OUT_DIR)),
}
with open(f"{OUT_DIR}/run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print(json.dumps(manifest, indent=2, default=str))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{
  "run_id": "20261116T092814Z",
  "spark_version": "3.5.5-amzn-1",
  "on_cluster": true,
  "inputs": {
    "prices": "s3://inf2006-g12-2500837/task2/cleaned/prices",
    "gdelt_daily_features": "s3://inf2006-g12-2500837/task2/cleaned/gdelt/daily_features"
  },
  "row_counts": {
    "prices": 605887,
    "gdelt_dates": 150,
    "scored_ticker_days": 595789,
    "returns_nulled_as_artefacts": 6,
    "q2_joined_days": 100
  },
  "parameters": {
    "window": 20,
    "z_high": 2.0,
    "z_low": -2.0,
    "max_abs_ret": 0.5,
    "groups": 3
  },
  "stage_seconds": {
    "load_and_check": 10.8,
    "q1_compute": 13.1,
    "q2_compute": 2.0
  },
  "total_seconds": 101.6,
  "outputs": [
    "q1_class_summary.csv",
    "q1_monthly_flags.csv",
    "q1_monthly_high_share.png",
    "q1_volume_class_and_zbucket.png",
    "q1_zbucket_summary.csv",
    "q2_block_composition.csv",
    "q2_controlled_by_today_vol.csv",
    "q2_joined_daily.csv",
    "q2_news_volume_groups.csv",
    "q2_next_day_vol_b

In [15]:
import os
print(os.getcwd())
print(os.listdir(OUT_DIR))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/mnt/yarn/usercache/livy/appcache/application_1794820524138_0002/container_1794820524138_0002_01_000001
['q1_class_summary.csv', 'q1_zbucket_summary.csv', 'q1_monthly_flags.csv', 'q1_volume_class_and_zbucket.png', 'q1_monthly_high_share.png', 'q2_news_volume_groups.csv', 'q2_tone_groups.csv', 'q2_controlled_by_today_vol.csv', 'q2_spearman_checks.csv', 'q2_joined_daily.csv', 'q2_block_composition.csv', 'q2_next_day_vol_by_news_group.png', 'q2_today_vs_next_day_vol.png', 'run_manifest.json']

In [16]:
import subprocess
r = subprocess.run(["aws","s3","cp","--recursive",OUT_DIR,
                    "s3://inf2006-g12-2500837/evidence/task4"],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

.csv
Completed 91.6 KiB/329.1 KiB (1.2 MiB/s) with 6 file(s) remaining
Completed 92.4 KiB/329.1 KiB (1.2 MiB/s) with 6 file(s) remaining
upload: evidence/task4/q1_zbucket_summary.csv to s3://inf2006-g12-2500837/evidence/task4/q1_zbucket_summary.csv
Completed 92.4 KiB/329.1 KiB (1.2 MiB/s) with 5 file(s) remaining
Completed 185.3 KiB/329.1 KiB (2.3 MiB/s) with 5 file(s) remaining
upload: evidence/task4/q1_monthly_high_share.png to s3://inf2006-g12-2500837/evidence/task4/q1_monthly_high_share.png
Completed 185.3 KiB/329.1 KiB (2.3 MiB/s) with 4 file(s) remaining
Completed 244.9 KiB/329.1 KiB (2.6 MiB/s) with 4 file(s) remaining
upload: evidence/task4/q2_next_day_vol_by_news_group.png to s3://inf2006-g12-2500837/evidence/task4/q2_next_day_vol_by_news_group.png
Completed 244.9 KiB/329.1 KiB (2.6 MiB/s) with 3 file(s) remaining
Completed 245.4 KiB/329.1 KiB (2.6 MiB/s) with 3 file(s) remaining
upload: evidence/task4/q2_tone_groups.csv to s3://inf2006-g12-2500837/evidence/task4/q2_tone_group